# Silver Positions

Loads data from `bundesliga-2022-2023.batch.bronze_positions` (enriched tracking data with attacking direction, normalized coordinates, ball distance, and previous-frame columns) and shows sample rows.

## Silver Transformation Steps

Input: `bronze_positions` (36 columns, \~23.7M rows) → Output: `silver_positions` (16 columns, \~3M rows)

| Step | Cell | Action | Columns Added / Modified |
| --- | --- | --- | --- |
| 0 | 3 | Load `bronze_positions` from Delta table | 36 input columns loaded |
| 1 | 4 | Filter out `referee` rows (862,572 rows removed) | No new columns (rows reduced to \~22.9M) |
| 2 | 5 | Group by `match_id`, `team_id`, `frame_id`, `game_section`, `timestamp`. Collect players into sorted `players` struct array (21 fields per player, sorted by `x_norm` ascending). | `players` added (col 6). 6 columns total, \~3M rows (1 row per team per frame). |
| 3 | 6 | Add `offside_line` = `players[1].x_norm` (second-lowest X = first outfield player) and `offside_line_perc` = `offside_line / pitch_x * 100`. Uses `create_map` for pitch_x lookup (no join). | `offside_line`, `offside_line_perc` added (cols 7–8) |
| 4 | 7 | Add `play_state` via window on `ball_status`: `1` → `"active"`, `0` → `"interruption"`. Propagated from BALL row to all teams at the same frame. | `play_state` added (col 9) |
| 5 | 8 | Add `has_possession` (boolean) via window on `ball_possession`: `1` → home team `True`, `2` → guest team `True`. Uses `create_map` for home/guest team ID lookup (no join). | `has_possession` added (col 10) |
| 6 | 9 | Add `possession_zone` — BALL's `pitch_zone` from window, mirrored for the opponent team (first-third ↔ final-third, left ↔ right). Possessing team and BALL keep raw zone. | `possession_zone` added (col 11) |
| 7 | 10 | Define target points: `target_point` (105, 34) = attacking goal line centre, `opp_target_point` (0, 34) = opponent goal line centre. Stored as Spark maps. | Temp: `target_point`, `opp_target_point` |
| 8 | 11 | Add `ball_distance_target`, `prev_ball_distance_target` — Euclidean distance from ball to target goal (attacking for possessing team, defending for opponent). Propagated via window. Add `weight_ball_distance_target`, `prev_weight_ball_distance_target` (inverse: `1/(1+d)`), and `frame_score` (weight diff). | `ball_distance_target`, `prev_ball_distance_target`, `weight_ball_distance_target`, `prev_weight_ball_distance_target`, `frame_score` added (cols 12–16) |

**Output schema (16 columns):**

| # | Column | Type | Source |
| --- | --- | --- | --- |
| 1–5 | `match_id`, `team_id`, `frame_id`, `game_section`, `timestamp` | various | Group keys from bronze |
| 6 | `players` | array\<struct\> | Step 2 — collect_list of 21 player fields, sorted by x_norm |
| 7–8 | `offside_line`, `offside_line_perc` | double | Step 3 — players[1].x_norm + create_map pitch_x |
| 9 | `play_state` | string | Step 4 — window on ball_status |
| 10 | `has_possession` | boolean | Step 5 — window on ball_possession + create_map home/guest |
| 11 | `possession_zone` | string | Step 6 — BALL pitch_zone + mirror map for opponent |
| 12–13 | `ball_distance_target`, `prev_ball_distance_target` | double | Step 8 — Euclidean to goal, window-propagated |
| 14–15 | `weight_ball_distance_target`, `prev_weight_ball_distance_target` | double | Step 8 — inverse distance `1/(1+d)` |
| 16 | `frame_score` | double | Step 8 — weight diff (current − prev) |

In [0]:
# ── Load bronze_positions and show sample rows ──

from pyspark.sql.functions import col

silver_df = spark.table("`bundesliga-2022-2023`.batch.bronze_positions")

print(f"Table: `bundesliga-2022-2023`.batch.bronze_positions")
print(f"  Columns: {len(silver_df.columns)}")
print(f"  Rows: {silver_df.count():,}")
print(f"  Schema:")
for f in silver_df.schema.fields:
    print(f"    {f.name}: {f.dataType}")

print("\n=== Sample: ball rows (first 10) ===")
display(
    silver_df.filter(col("team_id") == "BALL")
    .select("match_id", "frame_id", "timestamp", "x", "y", "z", "speed", "distance", "acceleration", "ball_possession", "ball_status", "attacking_direction", "x_norm", "y_norm", "ball_distance")
    .orderBy("match_id", "frame_id")
    .limit(10)
)

print("\n=== Sample: player rows (first 10) ===")
display(
    silver_df.filter(col("team_id") != "BALL")
    .filter(col("team_id") != "referee")
    .select("match_id", "frame_id", "timestamp", "team_id", "person_id", "x", "y", "speed", "attacking_direction", "x_norm", "y_norm", "ball_distance", "prev_x", "prev_y", "prev_speed")
    .orderBy("match_id", "frame_id")
    .limit(10)
)

print("\n=== Sample: all entities at one frame ===")
sample_frame = silver_df.filter(
    (col("match_id") == "DFL-MAT-J03WMX") &
    (col("frame_id") == 10001)
).select("team_id", "person_id", "x", "y", "x_norm", "y_norm", "attacking_direction", "ball_distance", "prev_x", "prev_y")
display(sample_frame.orderBy(col("team_id"), col("person_id")))
print(f"\nEntities at this frame: {sample_frame.count()}")

In [0]:
# ── Filter out referee rows ──

before = silver_df.count()
silver_df = silver_df.filter(col("team_id") != "referee")
after = silver_df.count()

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Removed:     {before - after:,} referee rows")

print("\n=== Remaining team_id values ===")
silver_df.groupBy("team_id").count().orderBy("team_id").show()

In [0]:
# ── Group by match_id, team_id, frame_id and collect players into a struct array ──
# Each row = one team at one frame, with all players nested in a `players` array of structs.

from pyspark.sql.functions import collect_list, struct, array_sort, col, expr

player_cols = [
    "person_id", "x", "y", "z", "speed", "distance", "acceleration",
    "x_norm", "y_norm", "ball_distance", "attacking_direction",
    "ball_possession", "ball_status",
    "prev_x", "prev_y", "prev_speed", "prev_ball_distance",
    "prev_x_norm", "prev_y_norm",
    "pitch_zone", "zone_id"
]

silver_grouped = silver_df.groupBy("match_id", "team_id", "frame_id", "game_section", "timestamp").agg(
    collect_list(struct(*[col(c) for c in player_cols])).alias("players")
).withColumn(
    "players", expr("array_sort(players, (a, b) -> CASE WHEN a.x_norm < b.x_norm THEN -1 WHEN a.x_norm > b.x_norm THEN 1 ELSE 0 END)")
)

print(f"Grouped rows: {silver_grouped.count():,}")
print(f"Columns: {len(silver_grouped.columns)}")
print(f"Schema:")
for f in silver_grouped.schema.fields:
    print(f"  {f.name}: {f.dataType}")

print("\n=== Sample: match DFL-MAT-J03WMX, frame 10001 ===")
display(
    silver_grouped.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10001)
    ).select("team_id", "timestamp", "players")
    .orderBy("team_id")
)

In [0]:
# ── Add offside_line and offside_line_perc ──
# players array is already sorted by x_norm ascending, so players[0] = GK (lowest x_norm).
# The offside line is the second-lowest x_norm = players[1].x_norm (first outfield player).
# offside_line_perc = offside_line / pitch_x * 100 (rounded to 2 decimals)

from pyspark.sql.functions import col, expr, element_at, create_map, lit, round

# Build pitch_x map from match_info (same approach as bronze)
match_info = spark.table("`bundesliga-2022-2023`.batch.match_info")
pitch_rows = match_info.select("match_id", "pitch_x").collect()
pitch_x_args = []
for r in pitch_rows:
    pitch_x_args += [lit(r["match_id"]), lit(r["pitch_x"])]
pitch_x_map = create_map(*pitch_x_args)

silver_grouped = silver_grouped.withColumn(
    "offside_line", expr("get(players, 1).x_norm")
).withColumn(
    "offside_line_perc",
    round(col("offside_line") / element_at(pitch_x_map, col("match_id")) * 100, 2)
)

print(f"Columns: {len(silver_grouped.columns)}")
print(f"  Added: offside_line, offside_line_perc")

print("\n=== Sample: match DFL-MAT-J03WMX, frame 10001 ===")
display(
    silver_grouped.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10001)
    ).select("team_id", "offside_line", "offside_line_perc")
    .orderBy("team_id")
)

In [0]:
# ── Add play_state: "active" or "interruption" from ball_status ──
# ball_status is carried inside the BALL row's players struct (players[0].ball_status).
# Uses a window function to propagate it to all teams at the same frame — no join needed.
# ball_status: 1 = alive (in play) → "active", 0 = dead (out of play) → "interruption"

from pyspark.sql.functions import col, when, max as spark_max, expr
from pyspark.sql.window import Window

w = Window.partitionBy("match_id", "frame_id")

silver_grouped = silver_grouped.withColumn(
    "play_state",
    when(
        spark_max(when(col("team_id") == "BALL", expr("get(players, 0).ball_status"))).over(w) == 1,
        "active"
    ).otherwise("interruption")
)

print(f"Columns: {len(silver_grouped.columns)}")
print(f"  Added: play_state (no join — uses window on ball distance)")

print("\n=== Distribution of play_state ===")
silver_grouped.groupBy("play_state").count().orderBy("play_state").show()

print("\n=== Sample: match DFL-MAT-J03WMX, frame 10001 ===")
display(
    silver_grouped.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10001)
    ).select("team_id", "play_state", "offside_line")
    .orderBy("team_id")
)

In [0]:
# ── Add has_possession: true/false based on ball_possession ──
# ball_possession is carried in the BALL row's struct: 1 = home, 2 = guest.
# Uses window to get ball_possession per frame, then create_map to resolve home/guest team_id — no join.

from pyspark.sql.functions import col, when, max as spark_max, expr, element_at, create_map, lit
from pyspark.sql.window import Window

w = Window.partitionBy("match_id", "frame_id")

# Get ball_possession from BALL row via window (1=home, 2=guest)
ball_poss = spark_max(when(col("team_id") == "BALL", expr("get(players, 0).ball_possession"))).over(w)

# Build home/guest team maps from match_info
mi_rows = spark.table("`bundesliga-2022-2023`.batch.match_info").select("match_id", "home_team_id", "guest_team_id").collect()
home_args, guest_args = [], []
for r in mi_rows:
    home_args += [lit(r["match_id"]), lit(r["home_team_id"])]
    guest_args += [lit(r["match_id"]), lit(r["guest_team_id"])]
home_map = create_map(*home_args)
guest_map = create_map(*guest_args)

silver_grouped = silver_grouped.withColumn(
    "has_possession",
    when(
        (ball_poss == 1) & (col("team_id") == element_at(home_map, col("match_id"))),
        True
    ).when(
        (ball_poss == 2) & (col("team_id") == element_at(guest_map, col("match_id"))),
        True
    ).otherwise(False)
)

print(f"Columns: {len(silver_grouped.columns)}")
print(f"  Added: has_possession")

print("\n=== Distribution of has_possession ===")
silver_grouped.groupBy("has_possession").count().orderBy("has_possession").show()

print("\n=== Sample: match DFL-MAT-J03WMX, frame 10001 ===")
display(
    silver_grouped.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10001)
    ).select("team_id", "play_state", "has_possession", "offside_line")
    .orderBy("team_id")
)

In [0]:
# ── Add possession_zone: pitch_zone of the BALL, adjusted per team perspective ──
# The ball's pitch_zone is in the POSSESSION team's coordinate system (bronze resolves
# BALL's attacking_direction from ball_possession).
# - Possessing team + BALL: keep raw zone (correct perspective)
# - Opponent team: mirror first-third ↔ final-third (second-third and Y lanes unchanged)

from pyspark.sql.functions import col, when, max as spark_max, expr, lit, create_map
from pyspark.sql.window import Window

w = Window.partitionBy("match_id", "frame_id")

# Raw zone from the BALL row (possession team's perspective)
raw_zone = spark_max(when(col("team_id") == "BALL", expr("get(players, 0).pitch_zone"))).over(w)

# Mirror map: flip BOTH X (first ↔ final) AND Y (left ↔ right) for opponent perspective
mirror_map = create_map(
    lit("first-third_left"),    lit("final-third_right"),
    lit("first-third_centre"),  lit("final-third_centre"),
    lit("first-third_right"),   lit("final-third_left"),
    lit("second-third_left"),   lit("second-third_right"),
    lit("second-third_centre"), lit("second-third_centre"),
    lit("second-third_right"),  lit("second-third_left"),
    lit("final-third_left"),    lit("first-third_right"),
    lit("final-third_centre"),  lit("first-third_centre"),
    lit("final-third_right"),   lit("first-third_left"),
)

# Opponent rows get mirrored zone; possessing team and BALL keep raw
silver_grouped = silver_grouped.withColumn(
    "possession_zone",
    when((col("has_possession") == False) & (col("team_id") != "BALL"), mirror_map[raw_zone])
    .otherwise(raw_zone)
)

print(f"Columns: {len(silver_grouped.columns)}")
print(f"  Added: possession_zone (mirrored for opponent)")

print("\n=== Distribution of possession_zone ===")
silver_grouped.groupBy("possession_zone").count().orderBy("possession_zone").show()

print("\n=== Sample: match DFL-MAT-J03WMX, frame 10001 ===")
display(
    silver_grouped.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10001)
    ).select("team_id", "play_state", "has_possession", "possession_zone", "offside_line")
    .orderBy("team_id")
)

In [0]:
# ── Define target point (attack-normalized coordinates) ──
# Target: x=105 (goal line), y=34 (centre)
# Opponent target mirrored across pitch: (PITCH_X - TARGET_X, TARGET_Y)
# Stored as Spark maps for use in distance calculations.

from pyspark.sql.functions import lit, create_map

TARGET_X = 105.0
TARGET_Y = 34.0
PITCH_X = 105.0

target_point = create_map(lit("x"), lit(TARGET_X), lit("y"), lit(TARGET_Y))
opp_target_point = create_map(lit("x"), lit(PITCH_X - TARGET_X), lit("y"), lit(TARGET_Y))

print("Target (attack-normalized):")
print(f"  x = {TARGET_X}  (goal line)")
print(f"  y = {TARGET_Y}  (pitch centre)")
print(f"  Spark map: {target_point}")
print(f"Opponent target (mirrored):")
print(f"  x = {PITCH_X - TARGET_X}  (opponent goal line)")
print(f"  y = {TARGET_Y}")
print(f"  Spark map: {opp_target_point}")


In [0]:
# ── Add ball_distance_target: distance from ball to the attacking penalty spot ──
# The ball's x_norm/y_norm are in the POSSESSION team's coordinate system.
# - Team in possession: their target is at (105, 34) = goal line centre
# - Team NOT in possession: their target is at (0, 34) = (105-105, 34) = opponent goal line
#   (mirrored in the possession team's coordinate system)
# Compute both distances on the BALL row, propagate via window, then select per team.

from pyspark.sql.functions import col, when, max as spark_max, expr, sqrt, lit, round
import pyspark.sql.functions as F
from pyspark.sql.window import Window

bx = expr("get(players, 0).x_norm")
by = expr("get(players, 0).y_norm")
pbx = expr("get(players, 0).prev_x_norm")
pby = expr("get(players, 0).prev_y_norm")

# Extract target coordinates from Spark maps (defined in previous cell)
tx = target_point.getItem("x")
ty = target_point.getItem("y")
ox = opp_target_point.getItem("x")
oy = opp_target_point.getItem("y")

# Compute both distances on the BALL row (current + prev)
silver_grouped = silver_grouped.withColumn(
    "_dist_attacking",
    when(col("team_id") == "BALL",
        round(sqrt((bx - tx) * (bx - tx) + (by - ty) * (by - ty)), 2)
    )
).withColumn(
    "_dist_defending",
    when(col("team_id") == "BALL",
        round(sqrt((bx - ox) * (bx - ox) + (by - oy) * (by - oy)), 2)
    )
).withColumn(
    "_prev_dist_attacking",
    when(col("team_id") == "BALL",
        round(sqrt((pbx - tx) * (pbx - tx) + (pby - ty) * (pby - ty)), 2)
    )
).withColumn(
    "_prev_dist_defending",
    when(col("team_id") == "BALL",
        round(sqrt((pbx - ox) * (pbx - ox) + (pby - oy) * (pby - oy)), 2)
    )
)

# Propagate all from BALL row to all team rows at the same frame
w = Window.partitionBy("match_id", "frame_id")
silver_grouped = silver_grouped.withColumn(
    "_dist_attacking", spark_max(col("_dist_attacking")).over(w)
).withColumn(
    "_dist_defending", spark_max(col("_dist_defending")).over(w)
).withColumn(
    "_prev_dist_attacking", spark_max(col("_prev_dist_attacking")).over(w)
).withColumn(
    "_prev_dist_defending", spark_max(col("_prev_dist_defending")).over(w)
)

# Select: possessing team uses attacking target, opponent uses defending, BALL uses attacking
silver_grouped = silver_grouped.withColumn(
    "ball_distance_target",
    when(col("has_possession") == True, col("_dist_attacking"))
    .when(col("team_id") == "BALL", col("_dist_attacking"))
    .otherwise(col("_dist_defending"))
).withColumn(
    "prev_ball_distance_target",
    when(col("has_possession") == True, col("_prev_dist_attacking"))
    .when(col("team_id") == "BALL", col("_prev_dist_attacking"))
    .otherwise(col("_prev_dist_defending"))
).drop("_dist_attacking", "_dist_defending", "_prev_dist_attacking", "_prev_dist_defending")

# Weight: inverse distance (1 / (1 + distance)) — closer to target = higher weight
silver_grouped = silver_grouped \
    .withColumn("weight_ball_distance_target", F.round(1.0 / (1.0 + col("ball_distance_target")), 4)) \
    .withColumn("prev_weight_ball_distance_target", F.round(1.0 / (1.0 + col("prev_ball_distance_target")), 4))

# Frame score: weight diff = current weight - prev weight
# Positive when ball approaching this team's target, negative when retiring
silver_grouped = silver_grouped \
    .withColumn("frame_score", F.round(col("weight_ball_distance_target") - col("prev_weight_ball_distance_target"), 6))

print(f"Columns: {len(silver_grouped.columns)}")
print(f"  Updated: ball_distance_target + prev_ball_distance_target + weight_ball_distance_target + prev_weight_ball_distance_target + frame_score")

print("\n=== Sample: match DFL-MAT-J03WMX, frame 10001 ===")
display(
    silver_grouped.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10001)
    ).select("team_id", "has_possession", "possession_zone", "ball_distance_target", "prev_ball_distance_target",
            "weight_ball_distance_target", "prev_weight_ball_distance_target", "frame_score")
    .orderBy("team_id")
)

## Why weight diffs are not specular between teams

The weight is `1 / (1 + distance)`, and the weight diff is `weight - prev_weight`.
When the ball approaches one team's target, that team's diff is positive and the
opponent's is negative (and vice versa) — but the **magnitudes don't match**.

### Reason 1: Nonlinear (convex) weight function

`1/(1+d)` has derivative `-1/(1+d)²` — **steeper near small d** (close to target)
and **flatter at large d** (far from target). Even when the ball moves perfectly
along y=34 so that `Δd_possession = -Δd_opponent`, the weight changes differ:

| | distance | prev | Δd | weight diff |
|---|---|---|---|---|
| Opponent (closer) | 41.1 | 41.68 | −0.58 | **+0.0004** |
| Possession (farther) | 41.9 | 41.32 | +0.58 | **−0.0003** |

Same distance change (±0.58), but the team closer to its target gets a larger
weight change because the function is steeper there.

### Reason 2: 2D geometry

The two targets are at (105,34) and (0,34). Only when the ball moves **along
y=34** do the distance changes sum to zero. When the ball moves diagonally,
`Δd_possession ≠ -Δd_opponent`, adding another source of asymmetry.

### Conclusion

The weight diffs are **always opposite in sign** but **rarely equal in magnitude**.
Equal magnitudes occur only when the ball is exactly equidistant from both targets
(d ≈ 52.5, the pitch midpoint) — where both derivatives of `1/(1+d)` coincide.

In [0]:
# ── silver_grouped summary ──

from pyspark.sql.functions import col, countDistinct

n_rows = silver_grouped.count()
n_cols = len(silver_grouped.columns)
n_matches = silver_grouped.select("match_id").distinct().count()
n_teams = silver_grouped.select("team_id").distinct().count()
n_frames = silver_grouped.select("frame_id").distinct().count()

print(f"=== silver_grouped summary ===")
print(f"  Rows:      {n_rows:,}")
print(f"  Columns:   {n_cols}")
print(f"  Matches:   {n_matches}")
print(f"  Teams:     {n_teams} (incl. BALL)")
print(f"  Frames:    {n_frames:,}")

print(f"\n=== Columns ({n_cols}) ===")
print(f"{'#':<4} {'Column':<25} {'Type':<45}")
print("-" * 76)
for i, f in enumerate(silver_grouped.schema.fields, 1):
    print(f"{i:<4} {f.name:<25} {str(f.dataType):<45}")

print(f"\n=== play_state distribution ===")
silver_grouped.groupBy("play_state").count().orderBy("play_state").show()

print(f"\n=== Rows per team ===")
silver_grouped.groupBy("team_id").count().orderBy("team_id").show()

In [0]:
# ── Save silver_grouped as Delta table ──
# Target: `bundesliga-2022-2023`.batch.silver_positions
# Partitioned by match_id for efficient per-match queries.

# Drop existing table first to avoid UC metadata conflicts with schema changes
spark.sql("DROP TABLE IF EXISTS `bundesliga-2022-2023`.batch.silver_positions")

silver_grouped.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("match_id") \
    .format("delta") \
    .saveAsTable("`bundesliga-2022-2023`.batch.silver_positions")

print(f"Saved to `bundesliga-2022-2023`.batch.silver_positions")
print(f"  Rows: {silver_grouped.count():,}")
print(f"  Columns: {len(silver_grouped.columns)}")

# Verify table exists and row count matches
saved = spark.table("`bundesliga-2022-2023`.batch.silver_positions")
saved_count = saved.count()
print(f"  Verified rows: {saved_count:,}")
print(f"  Match: {'\u2705' if saved_count == silver_grouped.count() else '\u274c'}")

In [0]:
# ── Verify row count: 3 x distinct (match_id, frame_id) ──
# Each frame should have exactly 3 rows: BALL + 2 teams.

from pyspark.sql.functions import col, count as spark_count

silver_grouped = spark.table("`bundesliga-2022-2023`.batch.silver_positions")

distinct_frames = silver_grouped.select("match_id", "frame_id").distinct().count()
expected = distinct_frames * 3
actual = silver_grouped.count()

print(f"=== Row count verification ===")
print(f"  Distinct (match_id, frame_id):  {distinct_frames:,}")
print(f"  Expected rows (3 x frames):    {expected:,}")
print(f"  Actual rows:                    {actual:,}")
print(f"  Match: {'\u2705' if expected == actual else '\u274c'}")

# Check if any frame has != 3 rows
rows_per_frame = silver_grouped.groupBy("match_id", "frame_id").agg(
    spark_count("*").alias("n_rows")
)
anomalies = rows_per_frame.filter(col("n_rows") != 3)
n_anomalies = anomalies.count()

print(f"\n=== Anomaly check ===")
print(f"  Frames with != 3 rows: {n_anomalies}")
if n_anomalies > 0:
    print(f"  Distribution of row counts per frame:")
    rows_per_frame.groupBy("n_rows").count().orderBy("n_rows").show()
    print(f"  Sample anomalies:")
    anomalies.orderBy("match_id", "frame_id").show(10)
else:
    print(f"  \u2705 All frames have exactly 3 rows (BALL + 2 teams)")